In [1]:
import json

In [2]:
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            if line.strip():  # skip empty lines
                data.append(json.loads(line))
    return data


languages = ['en', 'es', 'fr', 'hi', 'it']

In [3]:
for lang in languages:
    file_path = f'{lang}_train_data.jsonl'
    label_path = f'{lang}_train_label.jsonl'
    data = load_jsonl(file_path)
    labels = load_jsonl(label_path)
    print(f"Language: {lang}, Data samples: {len(data)}, Labels: {len(labels)}")

Language: en, Data samples: 24, Labels: 24
Language: es, Data samples: 20, Labels: 20
Language: fr, Data samples: 360, Labels: 360
Language: hi, Data samples: 265, Labels: 265
Language: it, Data samples: 360, Labels: 360


In [5]:
# print the keys of the first data sample and label for verification
if data:
    print(f"First data sample keys: {data[0].keys()}")
    print(f"First data sample label: {labels[0].keys()}")


First data sample keys: dict_keys(['index', 'title', 'abstract', 'doi', 'url', 'extracted', 'datafile', 'authors', 'question', 'model_id', 'model_config', 'prompt', 'output_text', 'output_tokens', 'output_logits'])
First data sample label: dict_keys(['index', 'has_fluency_mistakes', 'has_factual_mistakes'])


In [6]:
# print the first question and output_text for verification
if data and labels:
    print(f"First question: {data[0].get('question', 'N/A')}")
    print(f"First output_text: {data[0].get('output_text', 'N/A')}")
    print(f"First label: {labels[0]}")

First question: Quali dataset vengono usati per valutare i modelli addestrati, e come sono strutturati?
First output_text: L'articolo "Distributional Inclusion Hypothesis and Quantifications: Probing for Hypernymy in Functional Distributional Semantics" di Chun Hei Lo et al. utilizza diversi dataset per valutare i modelli addestrati. Ecco una descrizione dei dataset e della loro struttura:* **WordNet:** Questo dataset è una banca dati di sinonimi, antonimi, iperomini e iponimi per le parole inglesi. Viene utilizzato per definire le relazioni di ipernomia tra le parole, che sono poi utilizzate per valutare la capacità dei modelli di comprendere queste relazioni.* **FrameNet:** Questo dataset contiene informazioni sulle "frame", ovvero concetti astratti che rappresentano situazioni o eventi. Ogni frame è composto da "roles", ovvero entità che partecipano al frame. Viene utilizzato per valutare la capacità dei modelli di comprendere le relazioni semantiche tra parole in contesti specifici

In [ ]:
import requests
from urllib.parse import urlparse
import PyPDF2
import io
import re

def extract_pdf_text(url):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            pdf_file = io.BytesIO(response.content)
            pdf_reader = PyPDF2.PdfReader(pdf_file)
            text = ''
            for page in pdf_reader.pages:
                text += page.extract_text()
            return text
        return ''
    except:
        return ''

# Add PDF text to each item
for item in data:
    if 'url' in item and item['url'].endswith('.pdf'):
        item['pdf_text'] = extract_pdf_text(item['url'])
    else:
        item['pdf_text'] = ''
        
print(f"First item with PDF text: {data[0]}")


In [13]:
# print the keys of the first data sample and label for verification
if data:
    print(f"First data sample keys: {data[0].keys()}")
    print(f"First data sample label: {labels[0].keys()}")


First data sample keys: dict_keys(['index', 'title', 'abstract', 'doi', 'url', 'extracted', 'datafile', 'authors', 'question', 'model_id', 'model_config', 'prompt', 'output_text', 'output_tokens', 'output_logits', 'pdf_text'])
First data sample label: dict_keys(['index', 'has_fluency_mistakes', 'has_factual_mistakes'])


In [14]:
# check for cases where pdf_text is "" from data

empty_pdf_count = sum(1 for item in data if item.get('pdf_text', '') == '')
print(f"Number of items with empty pdf_text: {empty_pdf_count} out of {len(data)}")

Number of items with empty pdf_text: 0 out of 360


## Visualizing self check GPT results

In [4]:
file_path = 'selfcheck_results.jsonl'
results = load_jsonl(file_path)
print(type(results))

<class 'list'>


In [13]:
print(results[0])
predicted_list = [res['predicted_label'] for res in results]
# print the distribution of predicted labels
from collections import Counter
label_distribution = Counter(predicted_list)
print(label_distribution)

gt_labels = [res['factual_mistake'] for res in results]
gt_distribution = Counter(gt_labels)
print(gt_distribution)

{'index': 'en-train-0', 'prompt': 'In the article titled "Extrinsic Evaluation of Machine Translation Metrics" by Moghe,Nikita et al., what do the authors use extractive QA for in their methodology?', 'original_answer': 'According to the article "Extrinsic Evaluation of Machine Translation Metrics" by Moghe, Nikita et al., the authors use extractive QA (Question Answering) as a task to evaluate the quality of machine translation outputs. Specifically, they use extractive QA to assess the ability of machine translation systems to accurately answer questions about the source text, which is a measure of the system\'s ability to preserve the meaning and content of the original text.In their methodology, the authors use a dataset of questions and answers, where the questions are designed to test the understanding of the source text. They then translate the source text using different machine translation systems and evaluate the quality of the translations by assessing how well they can answ

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from pprint import pprint
# language wise evaluation
language_metrics = {}
for lang in languages:
    # basically we have to check the language from the first 2 characters of the id
    lang_results = [res for res in results if res['index'].startswith(lang[:2])]
    if not lang_results:
        continue
    y_true = [res['factual_mistake'] for res in lang_results]
    y_pred = [res['predicted_label'] for res in lang_results]   
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=1)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=1)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=1)
    language_metrics[lang] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

pprint(language_metrics)


{'en': {'accuracy': 0.3333333333333333,
        'f1_score': 0.37142857142857144,
        'precision': 0.4857142857142857,
        'recall': 0.3333333333333333},
 'es': {'accuracy': 0.3,
        'f1_score': 0.28585858585858587,
        'precision': 0.2956043956043956,
        'recall': 0.3},
 'fr': {'accuracy': 0.575,
        'f1_score': 0.6771415577524877,
        'precision': 0.8508711422002009,
        'recall': 0.575},
 'hi': {'accuracy': 0.5660377358490566,
        'f1_score': 0.6717282909130814,
        'precision': 0.8755635331440975,
        'recall': 0.5660377358490566},
 'it': {'accuracy': 0.37777777777777777,
        'f1_score': 0.3641203703703703,
        'precision': 0.4417063651611698,
        'recall': 0.37777777777777777}}


In [15]:
# print overall metrics
y_true = [res['factual_mistake'] for res in results]
y_pred = [res['predicted_label'] for res in results]
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=1
)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=
1)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=1
)
overall_metrics = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1
}
print("Overall Metrics:")
pprint(overall_metrics)

Overall Metrics:
{'accuracy': 0.49271137026239065,
 'f1_score': 0.5126460119519571,
 'precision': 0.5407687242122425,
 'recall': 0.4927113702623907}


## Visualizing xlm roberta results

In [4]:
data = load_jsonl('xlm_roberta_hallucination_results.json')
print(data[0].keys())

dict_keys(['index', 'title', 'abstract', 'doi', 'url', 'extracted', 'datafile', 'authors', 'question', 'model_id', 'model_config', 'prompt', 'output_text', 'output_tokens', 'output_logits', 'pdf_text', 'fluency_mistake', 'factual_mistake', 'hallucination_score', 'hallucinated'])


In [5]:
# print the distribution of predicted labels
from collections import Counter
label_distribution = Counter([item['hallucinated'] for item in data])
print(label_distribution)

Counter({'y': 982, 'n': 47})


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from pprint import pprint
# language wise evaluation
language_metrics = {}
for lang in languages:
    # basically we have to check the language from the first 2 characters of the id
    lang_results = [res for res in data if res['index'].startswith(lang[:2])]
    if not lang_results:
        continue
    y_true = [res['factual_mistake'] for res in lang_results]
    y_pred = [res['hallucinated'] for res in lang_results]   
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=1)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=1)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=1)
    language_metrics[lang] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }
pprint(language_metrics)

# print overall metrics
y_true = [res['factual_mistake'] for res in data]
y_pred = [res['hallucinated'] for res in data]
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=1)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=1)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=1)
overall_metrics = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1
}
print("Overall Metrics:")
pprint(overall_metrics)


{'en': {'accuracy': 0.6666666666666666,
        'f1_score': 0.6,
        'precision': 0.5454545454545455,
        'recall': 0.6666666666666666},
 'es': {'accuracy': 0.5,
        'f1_score': 0.36666666666666664,
        'precision': 0.28947368421052627,
        'recall': 0.5},
 'fr': {'accuracy': 0.9222222222222223,
        'f1_score': 0.890237636480411,
        'precision': 0.860397268777157,
        'recall': 0.9222222222222223},
 'hi': {'accuracy': 0.8,
        'f1_score': 0.8324023589629465,
        'precision': 0.8695711368277741,
        'recall': 0.8},
 'it': {'accuracy': 0.36666666666666664,
        'f1_score': 0.20302277432712215,
        'precision': 0.34575163398692804,
        'recall': 0.36666666666666664}}
Overall Metrics:
{'accuracy': 0.6822157434402333,
 'f1_score': 0.5901906672802858,
 'precision': 0.5317124404531869,
 'recall': 0.6822157434402333}
